# 📊 Introduction to RAG Evaluation

## Why Evaluation Is Hard in RAG

In regular ML, evaluation is straightforward:
```
Image classifier → accuracy on test set
```

In RAG, there are **two stages** that can each go wrong:

```
Stage 1: Retrieval — did we find the right documents?
Stage 2: Generation — did the LLM answer correctly given those documents?

Bad retrieval + Good generation → Wrong answer  ❌
Good retrieval + Bad generation → Wrong answer  ❌
Good retrieval + Good generation → Right answer ✅
```

You need to evaluate **both stages independently** to know where to improve.

## The RAG Evaluation Framework

```
                    ┌─────────────────────────┐
                    │    RAG System Output     │
                    └────────────┬────────────┘
                                 │
              ┌──────────────────┼──────────────────┐
              ▼                                     ▼
   ┌─────────────────────┐             ┌─────────────────────┐
   │  Retrieval Metrics  │             │ Generation Metrics  │
   │                     │             │                     │
   │  • Precision@K      │             │  • Faithfulness     │
   │  • Recall@K         │             │  • Answer relevance │
   │  • MRR              │             │  • Completeness     │
   │  • NDCG             │             │  • RAGAS score      │
   └─────────────────────┘             └─────────────────────┘
```

## What This Section Covers

| Notebook | What You Learn |
|---|---|
| `01_Retrieval_Metrics.ipynb` | Precision, Recall, MRR, NDCG |
| `02_Generation_Metrics.ipynb` | Faithfulness, answer relevance, RAGAS |
| `03_End_to_End_Evaluation.ipynb` | Test the full pipeline together |

In [1]:
# Quick demo: why you need BOTH retrieval AND generation metrics

# Scenario 1: Perfect retrieval, bad generation
retrieved_correct = [
    "Dropout prevents overfitting by randomly dropping neurons.",
    "Dropout rate is typically 0.2–0.5.",
]
generated_answer_bad = "Dropout is a type of data augmentation used to add noise to images."

print("Scenario 1: Good retrieval, bad generation")
print(f"  Retrieved: {retrieved_correct[0][:50]}...")
print(f"  Generated: {generated_answer_bad}")
print(f"  Retrieval OK? ✅   Generation OK? ❌")

print()

# Scenario 2: Bad retrieval, lucky generation
retrieved_wrong = [
    "The Eiffel Tower is 330 meters tall.",
]
generated_answer_lucky = "Dropout prevents overfitting by dropping neurons during training."

print("Scenario 2: Bad retrieval, LLM got lucky from training memory")
print(f"  Retrieved: {retrieved_wrong[0]}")
print(f"  Generated: {generated_answer_lucky}")
print(f"  Retrieval OK? ❌   Generation OK? ✅ (but unreliable — came from LLM memory!)")

print()
print("💡 Without evaluating BOTH, you can't tell what's broken.")

Scenario 1: Good retrieval, bad generation
  Retrieved: Dropout prevents overfitting by randomly dropping ...
  Generated: Dropout is a type of data augmentation used to add noise to images.
  Retrieval OK? ✅   Generation OK? ❌

Scenario 2: Bad retrieval, LLM got lucky from training memory
  Retrieved: The Eiffel Tower is 330 meters tall.
  Generated: Dropout prevents overfitting by dropping neurons during training.
  Retrieval OK? ❌   Generation OK? ✅ (but unreliable — came from LLM memory!)

💡 Without evaluating BOTH, you can't tell what's broken.


In [2]:
# What you need to run evaluation:

evaluation_inputs = {
    "question": "What is dropout?",
    "ground_truth_answer": "Dropout randomly deactivates neurons during training to prevent overfitting.",
    "ground_truth_doc_ids": ["doc_42", "doc_17"],  # Which docs should be retrieved

    # What the RAG system actually produced:
    "retrieved_doc_ids": ["doc_42", "doc_88", "doc_17"],
    "generated_answer": "Dropout is a technique that randomly drops neurons during training.",
}

print("Evaluation inputs needed:")
for k, v in evaluation_inputs.items():
    print(f"  {k}: {v}")

print("\n💡 The hardest part of RAG evaluation is building a labeled test set.")
print("   You need questions + correct answers + which docs should be retrieved.")

Evaluation inputs needed:
  question: What is dropout?
  ground_truth_answer: Dropout randomly deactivates neurons during training to prevent overfitting.
  ground_truth_doc_ids: ['doc_42', 'doc_17']
  retrieved_doc_ids: ['doc_42', 'doc_88', 'doc_17']
  generated_answer: Dropout is a technique that randomly drops neurons during training.

💡 The hardest part of RAG evaluation is building a labeled test set.
   You need questions + correct answers + which docs should be retrieved.


## Quick Evaluation Vocabulary

Before diving in, let's define the terms you'll see:

```
Relevant docs:  The docs that SHOULD be retrieved for a query (ground truth)
Retrieved docs: The docs your system ACTUALLY retrieved

True Positive:  Retrieved AND relevant
False Positive: Retrieved but NOT relevant
False Negative: Relevant but NOT retrieved

Precision: Of what I retrieved, how much was relevant?
           TP / (TP + FP)

Recall:    Of what's relevant, how much did I find?
           TP / (TP + FN)
```

---
Next: `01_Retrieval_Metrics.ipynb`